# 0.0 — Descarga y preparación de datasets

Descarga los tres datasets públicos de bootstrap desde Google Drive y los prepara
para el pipeline de entrenamiento YOLO:

- **Sección 1 — TACO_TN_UAV_2**: YOLOv8 nativo, sin conversión; aplica remapeo de clases.
- **Sección 2 — UAVVaste**: formato COCO → conversión a YOLO con `convert_coco`.
- **Sección 3 — TACO**: formato COCO → conversión a YOLO con `convert_coco`.
- **Sección 4 — Dataset propio del dron**: celda TODO condicionada a `gdrive_id` no vacío.

**Autor:** Jorge Ceferino Valdez — Maestría en Informática y Sistemas (UNPA)

In [1]:
import json
import os
import sys
import zipfile
from pathlib import Path

import gdown
import yaml

In [2]:
# Agrego la raíz del proyecto al sys.path para poder importar src/
project_root = Path(os.path.abspath("")).parent
if project_root not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"project_root: {project_root}")

project_root: /home/jorge/development/drone-waste-detect-compress


In [3]:
from src.config import config, external_datasets, project_dir, external_data_dir

print("Datasets externos configurados:")
for name, ds in external_datasets.items():
    print(f"  {name}: dest={ds['dest']!r}, format={ds['format']!r}, gdrive_id={ds['gdrive_id']!r}")

Datasets externos configurados:
  taco_tn_uav_2: dest='data/external/TACO_TN_UAV_2', format='yolov8_native', gdrive_id='19xJYrkmv7HnwNIIwA6a4BBdu6YX4ZbpH'
  uavvaste: dest='data/external/UAVVaste', format='coco', gdrive_id='1wK7CzaT3eIHPCe6YKMfteQYcK2madKwP'
  taco: dest='data/external/TACO', format='coco', gdrive_id='1YoFb_1MHwDTKVqvtksbjApCqF2pkHFKC'
  drone_captures: dest='data/raw/drone_captures/data', format='raw', gdrive_id=''


In [4]:
def download_and_extract(name: str, ds: dict) -> None:
    """Descarga el zip desde Google Drive y lo descomprime en ds['dest']."""
    gdrive_id = ds["gdrive_id"]
    dest_dir  = project_dir() / ds["dest"]
    dest_dir.mkdir(parents=True, exist_ok=True)

    zip_path = dest_dir.parent / f"{name}.zip"

    print(f"[{name}] Descargando desde Google Drive ID: {gdrive_id}")
    url = f"https://drive.google.com/uc?id={gdrive_id}"
    gdown.download(url, str(zip_path), quiet=False)

    print(f"[{name}] Descomprimiendo en {dest_dir} ...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(dest_dir)

    zip_path.unlink()
    print(f"[{name}] Listo. Zip eliminado.")

---
## Sección 1 — TACO_TN_UAV_2 (YOLOv8 nativo, sin conversión)

Dataset de residuos aéreos con 6460 imágenes y 9 clases, en formato YOLOv8 nativo.
Los splits train/valid/test ya están separados; solo se descarga, verifica y aplica
el remapeo de clases del config para generar `data_remapped.yaml`.

**Limitaciones del mapeo:**
- `aluminium wrap` → `film_bolsa`: aproximación; aluminium wrap es papel aluminio, no bolsa plástica.
- `red_pesca` no tiene equivalente en ningún dataset público.

Conteo esperado: train 4527 imágenes · valid 654 · test 1279

In [5]:
ds_taco_uav = external_datasets["taco_tn_uav_2"]
dest_taco_uav = project_dir() / ds_taco_uav["dest"]

if not dest_taco_uav.exists() or not any(dest_taco_uav.iterdir()):
    download_and_extract("taco_tn_uav_2", ds_taco_uav)
else:
    print(f"[taco_tn_uav_2] Ya existe en disco: {dest_taco_uav}")

[taco_tn_uav_2] Ya existe en disco: /home/jorge/development/drone-waste-detect-compress/data/external/TACO_TN_UAV_2


In [6]:
# Verificar estructura YOLOv8: train/images, train/labels, valid/images, valid/labels, test/images, test/labels, data.yaml
expected_dirs = [
    "train/images", "train/labels",
    "valid/images", "valid/labels",
    "test/images",  "test/labels",
]

print("Verificando estructura de TACO_TN_UAV_2:")
all_ok = True
for rel in expected_dirs:
    p = dest_taco_uav / rel
    exists = p.exists()
    n = sum(1 for f in p.glob("*") if f.is_file()) if exists else 0
    status = "OK" if exists else "FALTA"
    if not exists:
        all_ok = False
    print(f"  [{status}] {rel}/ ({n} archivos)")

data_yaml = dest_taco_uav / "data.yaml"
status = "OK" if data_yaml.exists() else "FALTA"
if not data_yaml.exists():
    all_ok = False
print(f"  [{status}] data.yaml")

assert all_ok, "Estructura de TACO_TN_UAV_2 incompleta"

# Conteo por split
print()
print("Imágenes por split:")
for split in ["train", "valid", "test"]:
    imgs = list((dest_taco_uav / split / "images").glob("*"))
    print(f"  {split}: {len(imgs)} imágenes")

Verificando estructura de TACO_TN_UAV_2:
  [OK] train/images/ (4527 archivos)
  [OK] train/labels/ (4527 archivos)
  [OK] valid/images/ (654 archivos)
  [OK] valid/labels/ (654 archivos)
  [OK] test/images/ (1279 archivos)
  [OK] test/labels/ (1279 archivos)
  [OK] data.yaml

Imágenes por split:
  train: 4527 imágenes
  valid: 654 imágenes
  test: 1279 imágenes


In [7]:
# Leer data.yaml original y mostrar las 9 clases
with open(data_yaml) as f:
    original_yaml = yaml.safe_load(f)

print("Clases originales en data.yaml:")
for i, name in enumerate(original_yaml.get("names", [])):
    print(f"  {i}: {name}")

Clases originales en data.yaml:
  0: aluminium wrap
  1: cardboard
  2: cigarette
  3: general waste
  4: glass
  5: metal
  6: paper
  7: plastic
  8: styrofoam


In [8]:
# Construir data_remapped.yaml con las 7 clases del proyecto
# Las 9 clases originales se mapean a las 7 clases del proyecto;
# varias clases originales colapsan a 'otros'.

project_classes = config["data"]["classes"]  # ['plastico_rigido', 'film_bolsa', ...]
class_mapping   = ds_taco_uav["class_mapping"]  # dict: nombre_original -> nombre_proyecto

original_names = ds_taco_uav["names"]  # lista de 9 clases originales

# Mapeo de ID original → ID proyecto
id_mapping = {}
print("Mapeo de clases (original -> proyecto):")
for orig_id, orig_name in enumerate(original_names):
    proj_name = class_mapping[orig_name]
    proj_id   = project_classes.index(proj_name)
    id_mapping[orig_id] = proj_id
    print(f"  {orig_id}: {orig_name!r:20s} → {proj_id}: {proj_name!r}")

print()
print("Nota: aluminium wrap -> film_bolsa es una aproximación (papel aluminio ≠ bolsa plástica)")
print("Nota: red_pesca no tiene equivalente en TACO_TN_UAV_2")

Mapeo de clases (original -> proyecto):
  0: 'aluminium wrap'     → 1: 'film_bolsa'
  1: 'cardboard'          → 6: 'otros'
  2: 'cigarette'          → 6: 'otros'
  3: 'general waste'      → 6: 'otros'
  4: 'glass'              → 4: 'vidrio'
  5: 'metal'              → 3: 'metal'
  6: 'paper'              → 6: 'otros'
  7: 'plastic'            → 0: 'plastico_rigido'
  8: 'styrofoam'          → 2: 'poliestireno'

Nota: aluminium wrap -> film_bolsa es una aproximación (papel aluminio ≠ bolsa plástica)
Nota: red_pesca no tiene equivalente en TACO_TN_UAV_2


In [9]:
# Guardar data_remapped.yaml
remapped_yaml_path = dest_taco_uav / "data_remapped.yaml"

remapped_yaml = {
    "path": str(dest_taco_uav),
    "train": "train/images",
    "val":   "valid/images",
    "test":  "test/images",
    "nc":    len(project_classes),
    "names": project_classes,
    # id_mapping como comentario embebido en el yaml
    "_class_mapping_note": (
        "9 clases originales mapeadas a 7 clases del proyecto. "
        "aluminium_wrap->film_bolsa es aproximacion; red_pesca sin equivalente."
    ),
}

with open(remapped_yaml_path, "w") as f:
    yaml.dump(remapped_yaml, f, allow_unicode=True, sort_keys=False)

print(f"data_remapped.yaml guardado en: {remapped_yaml_path}")
print()
print("Contenido:")
with open(remapped_yaml_path) as f:
    print(f.read())

data_remapped.yaml guardado en: /home/jorge/development/drone-waste-detect-compress/data/external/TACO_TN_UAV_2/data_remapped.yaml

Contenido:
path: /home/jorge/development/drone-waste-detect-compress/data/external/TACO_TN_UAV_2
train: train/images
val: valid/images
test: test/images
nc: 7
names:
- plastico_rigido
- film_bolsa
- poliestireno
- metal
- vidrio
- red_pesca
- otros
_class_mapping_note: 9 clases originales mapeadas a 7 clases del proyecto. aluminium_wrap->film_bolsa
  es aproximacion; red_pesca sin equivalente.



---
## Sección 2 — UAVVaste (COCO → YOLO)

Dataset de residuos UAV con 772 imágenes en formato COCO.

**Estructura real del dataset:**
- Imágenes: `images/` (directorio flat, 775 archivos)
- Anotaciones: `annotations/instances_default.json`

Se convierte con `convert_coco` de Ultralytics (`use_segments=False` → bounding boxes).

In [10]:
ds_uavvaste = external_datasets["uavvaste"]
dest_uavvaste = project_dir() / ds_uavvaste["dest"]

if not dest_uavvaste.exists() or not any(dest_uavvaste.iterdir()):
    download_and_extract("uavvaste", ds_uavvaste)
else:
    print(f"[uavvaste] Ya existe en disco: {dest_uavvaste}")

[uavvaste] Ya existe en disco: /home/jorge/development/drone-waste-detect-compress/data/external/UAVVaste


In [11]:
print(ds_uavvaste)
print(dest_uavvaste)

{'gdrive_id': '1wK7CzaT3eIHPCe6YKMfteQYcK2madKwP', 'dest': 'data/external/UAVVaste', 'format': 'coco', 'annotations': 'annotations/instances_default.json', 'images_dir': 'images'}
/home/jorge/development/drone-waste-detect-compress/data/external/UAVVaste


In [12]:
# Verificar estructura pre-conversión
images_dir = dest_uavvaste / ds_uavvaste["images_dir"]  # "images"
anno_file  = dest_uavvaste / ds_uavvaste["annotations"]  # "annotations/instances_default.json"

n_images = sum(1 for f in images_dir.glob("*") if f.is_file()) if images_dir.exists() else 0

print(f"images/  : {'OK' if images_dir.exists() else 'FALTA'} ({n_images} archivos)")
print(f"anno JSON: {'OK' if anno_file.exists() else 'FALTA'} ({anno_file.name})")

assert images_dir.exists(), f"Directorio de imágenes no encontrado: {images_dir}"
assert anno_file.exists(),  f"Archivo de anotaciones no encontrado: {anno_file}"

images/  : OK (773 archivos)
anno JSON: OK (instances_default.json)


In [ ]:
# from ultralytics.data.converter import convert_coco

# # convert_coco espera el directorio que contiene el JSON.
# # Para UAVVaste: labels_dir = dest/annotations/
# anno_dir = anno_file #.parent

# print(f"[uavvaste] Convirtiendo COCO → YOLO desde {anno_dir} ...")
# convert_coco(
#     labels_dir=str(anno_dir),
#     use_segments=False,
#     use_keypoints=False,
#     cls91to80=False,
# )
# print("[uavvaste] Conversión completada.")

In [13]:
from ultralytics.data.converter import convert_coco
from collections import defaultdict

def convert_coco_single(anno_file: Path, save_dir: Path):
    save_dir.mkdir(parents=True, exist_ok=True)
   
    with open(anno_file) as f:
        data = json.load(f)

    cats = {c["id"]: i for i, c in enumerate(data["categories"])}
    images = {x["id"]: x for x in data["images"]}
    
    ann_by_image = defaultdict(list)
    for ann in data["annotations"]:
        ann_by_image[ann["image_id"]].append(ann)

    for img_id, img_info in images.items():
        W, H = img_info["width"], img_info["height"]
        out_path = save_dir / Path(img_info["file_name"]).with_suffix(".txt")
        out_path.parent.mkdir(parents=True, exist_ok=True)
      
        lines = []
        for ann in ann_by_image[img_id]:
            cls = cats[ann["category_id"]]
            x, y, w, h = ann["bbox"]
            xc = (x + w / 2) / W
            yc = (y + h / 2) / H
            lines.append(f"{cls} {xc:.6f} {yc:.6f} {w/W:.6f} {h/H:.6f}")
        
        out_path.write_text("\n".join(lines))
    
    print(f"Convertidas {len(images)} imágenes → {save_dir}")

# convert_coco_single(
#     anno_file=Path("../data/external/TACO/annotations.json"),
#     save_dir=Path("../data/labels/TACO"),
# )

In [19]:
dir_destino = dest_uavvaste / "labels"
convert_coco_single(
    anno_file=Path(anno_file),
    save_dir=Path(dir_destino),
)

Convertidas 772 imágenes → /home/jorge/development/drone-waste-detect-compress/data/external/UAVVaste/labels


In [ ]:
# from ultralytics.data.converter import convert_coco
# from collections import defaultdict

# def convert_coco_single(anno_file: Path, save_dir: Path):
#     save_dir.mkdir(parents=True, exist_ok=True)
    
#     with open(anno_file) as f:
#         data = json.load(f)

#     cats = {c["id"]: i for i, c in enumerate(data["categories"])}
#     images = {x["id"]: x for x in data["images"]}
    
#     ann_by_image = defaultdict(list)
#     for ann in data["annotations"]:
#         ann_by_image[ann["image_id"]].append(ann)

#     for img_id, img_info in images.items():
#         W, H = img_info["width"], img_info["height"]
#         out_path = save_dir / Path(img_info["file_name"]).with_suffix(".txt")
#         out_path.parent.mkdir(parents=True, exist_ok=True)
        
#         lines = []
#         for ann in ann_by_image[img_id]:
#             cls = cats[ann["category_id"]]
#             x, y, w, h = ann["bbox"]
#             xc = (x + w / 2) / W
#             yc = (y + h / 2) / H
#             lines.append(f"{cls} {xc:.6f} {yc:.6f} {w/W:.6f} {h/H:.6f}")
        
#         out_path.write_text("\n".join(lines))
    
#     print(f"Convertidas {len(images)} imágenes → {save_dir}")

# # Uso:
# convert_coco_single(
#     anno_file=anno_file,          # Path al annotations.json
#     save_dir=Path("../data/labels/UAVVaste"),
# )


In [ ]:
# # convert_coco escribe los .txt en labels/ dentro del save_dir por defecto
# # (directorio hermano del labels_dir pasado, llamado 'labels').
# # Buscar los .txt generados.
# labels_out = dest_uavvaste / "labels"
# if not labels_out.exists():
#     # Ultralytics puede escribir en un subdirectorio relativo al cwd
#     labels_out = Path("labels")

# txt_files = list(labels_out.rglob("*.txt")) if labels_out.exists() else []
# print(f"[uavvaste] Archivos .txt generados: {len(txt_files)}")
# print(f"  Directorio de salida: {labels_out.resolve()}")
# if txt_files:
#     print(f"  Ejemplo: {txt_files[0]}")
#     print(f"  Contenido del primero (5 líneas):")
#     with open(txt_files[0]) as f:
#         for line in list(f)[:5]:
#             print(f"    {line.rstrip()}")

[uavvaste] Archivos .txt generados: 0
  Directorio de salida: /home/jorge/development/drone-waste-detect-compress/notebooks/labels


---
## Sección 3 — TACO (COCO → YOLO)

Dataset general de residuos con ~5200 imágenes en formato COCO.

**Estructura real del dataset:**
- Imágenes: organizadas en subdirectorios `batch_1/` a `batch_13/` (1503 archivos en total)
- Anotaciones: `annotations.json` en la raíz del dataset

Se convierte con `convert_coco` pasando el directorio raíz como `labels_dir`.

In [20]:
ds_taco = external_datasets["taco"]
dest_taco = project_dir() / ds_taco["dest"]

if not dest_taco.exists() or not any(dest_taco.iterdir()):
    download_and_extract("taco", ds_taco)
else:
    print(f"[taco] Ya existe en disco: {dest_taco}")

[taco] Ya existe en disco: /home/jorge/development/drone-waste-detect-compress/data/external/TACO


In [21]:
# Verificar estructura pre-conversión
anno_file_taco = dest_taco / ds_taco["annotations"]  # "annotations.json"

# Contar imágenes en todos los subdirectorios batch_*/
all_imgs = list(dest_taco.rglob("*.jpg")) + list(dest_taco.rglob("*.JPG")) + list(dest_taco.rglob("*.png"))
batch_dirs = sorted([d for d in dest_taco.iterdir() if d.is_dir() and d.name.startswith("batch_")])

print(f"annotations.json: {'OK' if anno_file_taco.exists() else 'FALTA'}")
print(f"Total imágenes:   {len(all_imgs)}")
print(f"Subdirectorios batch_*: {len(batch_dirs)}")
for bd in batch_dirs:
    n = sum(1 for f in bd.glob("*") if f.is_file())
    print(f"  {bd.name}/: {n} archivos")

assert anno_file_taco.exists(), f"Archivo de anotaciones no encontrado: {anno_file_taco}"

annotations.json: OK
Total imágenes:   1500
Subdirectorios batch_*: 15
  batch_1/: 101 archivos
  batch_10/: 100 archivos
  batch_11/: 100 archivos
  batch_12/: 100 archivos
  batch_13/: 100 archivos
  batch_14/: 100 archivos
  batch_15/: 85 archivos
  batch_2/: 92 archivos
  batch_3/: 97 archivos
  batch_4/: 89 archivos
  batch_5/: 112 archivos
  batch_6/: 97 archivos
  batch_7/: 127 archivos
  batch_8/: 100 archivos
  batch_9/: 100 archivos


In [ ]:
# Para TACO: las anotaciones están en la raíz (annotations.json),
# por lo que labels_dir es la raíz del dataset.
# print(f"[taco] Convirtiendo COCO → YOLO desde {dest_taco} ...")
# convert_coco(
#     labels_dir=str(dest_taco),
#     use_segments=False,
#     use_keypoints=False,
#     cls91to80=False,
# )
# print("[taco] Conversión completada.")

In [ ]:
# list(Path("../data/external/TACO").rglob("*.json"))

In [ ]:
# TACO
# convert_coco_single(
#     anno_file=Path("../data/external/TACO/annotations.json"),
#     save_dir=Path("../data/labels/TACO"),
# )


In [ ]:
# TACO
dir_destino_taco = dest_taco / "labels"
convert_coco_single(
    anno_file=Path(anno_file_taco),
    save_dir=Path(dir_destino_taco),
)

Convertidas 1500 imágenes → /home/jorge/development/drone-waste-detect-compress/data/external/TACO/labels


In [23]:
# Verificar archivos .txt generados
labels_out_taco = dest_taco / "labels"
if not labels_out_taco.exists():
    labels_out_taco = Path("labels")

txt_files_taco = list(labels_out_taco.rglob("*.txt")) if labels_out_taco.exists() else []
print(f"[taco] Archivos .txt generados: {len(txt_files_taco)}")
print(f"  Directorio de salida: {labels_out_taco.resolve()}")
if txt_files_taco:
    print(f"  Ejemplo: {txt_files_taco[0]}")
    print(f"  Contenido del primero (5 líneas):")
    with open(txt_files_taco[0]) as f:
        for line in list(f)[:5]:
            print(f"    {line.rstrip()}")

[taco] Archivos .txt generados: 1500
  Directorio de salida: /home/jorge/development/drone-waste-detect-compress/data/external/TACO/labels
  Ejemplo: /home/jorge/development/drone-waste-detect-compress/data/external/TACO/labels/batch_3/IMG_4897.txt
  Contenido del primero (5 líneas):
    36 0.626430 0.562347 0.096814 0.093444
    36 0.555556 0.515319 0.079248 0.038603
    36 0.744485 0.833487 0.073121 0.064032


---
## Sección 4 — Dataset propio del dron (TODO)

Esta celda descarga el dataset capturado con el DJI Mini 4 Pro sobre la costa
del Golfo San Jorge. La lógica está completamente escrita; se ejecuta automáticamente
en cuanto `external_datasets.drone_captures.gdrive_id` deje de estar vacío en `src/config.yaml`.

In [ ]:
# TODO: dataset propio del dron
# Completar gdrive_id en src/config.yaml → external_datasets.drone_captures.gdrive_id
# cuando el dataset esté subido a Google Drive.

drone_cfg = external_datasets["drone_captures"]
drone_gdrive_id = drone_cfg["gdrive_id"]

if not drone_gdrive_id:
    print("[drone_captures] gdrive_id vacío en config.yaml — descarga omitida.")
    print("  Para activar: completar external_datasets.drone_captures.gdrive_id en src/config.yaml.")
else:
    dest_dir = project_dir() / drone_cfg["dest"]
    dest_dir.mkdir(parents=True, exist_ok=True)
    zip_path = dest_dir.parent / "drone_captures.zip"

    print(f"[drone_captures] Descargando desde Google Drive ID: {drone_gdrive_id}")
    url = f"https://drive.google.com/uc?id={drone_gdrive_id}"
    gdown.download(url, str(zip_path), quiet=False)

    print(f"[drone_captures] Descomprimiendo en {dest_dir} ...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(dest_dir)

    zip_path.unlink()

    n_files = sum(1 for f in dest_dir.rglob("*") if f.is_file())
    print(f"[drone_captures] Listo. {n_files} archivos en {dest_dir}")
    assert n_files > 0, "La carpeta del dron está vacía tras la descarga."

---
## Verificación final

In [ ]:
print("=" * 60)
print("VERIFICACIÓN FINAL — artefactos esperados en disco")
print("=" * 60)

checks = [
    ("TACO_TN_UAV_2 descargado",      dest_taco_uav.exists()),
    ("TACO_TN_UAV_2 train/images",    (dest_taco_uav / "train/images").exists()),
    ("TACO_TN_UAV_2 valid/images",    (dest_taco_uav / "valid/images").exists()),
    ("TACO_TN_UAV_2 test/images",     (dest_taco_uav / "test/images").exists()),
    ("TACO_TN_UAV_2 data.yaml",       (dest_taco_uav / "data.yaml").exists()),
    ("TACO_TN_UAV_2 data_remapped.yaml", remapped_yaml_path.exists()),
    ("UAVVaste descargado",           dest_uavvaste.exists()),
    ("UAVVaste images/",              (dest_uavvaste / "images").exists()),
    ("UAVVaste anotaciones JSON",     (dest_uavvaste / "annotations" / "instances_default.json").exists()),
    ("TACO descargado",              dest_taco.exists()),
    ("TACO annotations.json",        (dest_taco / "annotations.json").exists()),
]

all_ok = True
for label, ok in checks:
    status = "OK" if ok else "FALTA"
    if not ok:
        all_ok = False
    print(f"  [{status}] {label}")

# Drone: solo si gdrive_id configurado
drone_gdrive_id = external_datasets["drone_captures"]["gdrive_id"]
if drone_gdrive_id:
    dest_drone = project_dir() / external_datasets["drone_captures"]["dest"]
    ok = dest_drone.exists() and any(dest_drone.rglob("*"))
    status = "OK" if ok else "FALTA"
    if not ok:
        all_ok = False
    print(f"  [{status}] drone_captures descargado")
else:
    print("  [SKIP] drone_captures — gdrive_id vacío, pendiente")

print()
if all_ok:
    print("Notebook 0.0 completado correctamente.")
else:
    print("Algunos artefactos faltan — revisar errores arriba.")